![](img/banner.png)

# SAT-Table: truth tables, recursion, and counting clusterings

**Mahmood Amintoosi, Fall 2026**

Computer Science Dept, Ferdowsi University of Mashhad


## Why this notebook?

Clustering is the task of splitting $n$ data points into $k$ groups. Before we
can *search* for the best split we must be able to *list* the candidates, and the
simplest candidate space in discrete mathematics is the set of $0/1$ vectors of
length $n$: the rows of a **truth table**.

Three things are done here, in this order:

1. write all rows of a truth table, first with nested loops and then with a
   **recursive** function;
2. repair the two mistakes that everybody makes in recursive enumeration
   (`return` in place of `yield`, and storing references to one shared array);
3. count the objects being enumerated with **Stirling numbers of the second
   kind**, and watch the count explode.


## 1. Three variables, three loops

For three Boolean variables $x_1,x_2,x_3$ the truth table has
$2\cdot 2\cdot 2 = 8$ rows, and three nested loops print them all:


In [1]:
for x1 in range(2):
    for x2 in range(2):
        for x3 in range(2):
            print(x1, x2, x3)


0 0 0
0 0 1
0 1 0
0 1 1
1 0 0
1 0 1
1 1 0
1 1 1


For four variables we would add a fourth loop; for ten variables, ten loops. The
data sets of this course have thousands of points, so this style of programming
does not scale. What we need is *one* loop that decides the first variable and
then hands the rest of the job to the same code. A function that calls itself is
called **recursive**.

## 2. A recursive function

Every recursive function has two parts:

* the **base case**: the smallest instance of the problem, solved directly.
  Here it is "all $n$ decisions are made", that is `i == n`; we then record the
  row;
* the **recursive case**: make one decision (fix `x[i]`), then call the function
  on the smaller problem (decide `x[i+1], x[i+2], \ldots`).

Two rules keep such a function correct: every call must get *closer* to the base
case (here `i` grows), and every branch must reach the base case.

Trace $n=2$ by hand. `SAT_Table(x, 0)` sets `x[0] = 0` and calls
`SAT_Table(x, 1)`; that call sets `x[1] = 0` and calls `SAT_Table(x, 2)`, whose
base case prints `[0, 0]`. Back at level 1 the loop moves on, `x[1] = 1` gives
`[0, 1]`, and level 1 is finished. Back at level 0 the loop moves on, `x[0] = 1`,
and the same two rows appear with a leading 1: four rows in total, $2^2$.


In [2]:
def SAT_Table(x, i):
    """Print every truth-table row that agrees with x on the first i entries."""
    if i == len(x):            # base case: nothing left to decide
        print(x)
    else:                      # recursive case: decide x[i], then the rest
        for x[i] in range(2):
            SAT_Table(x, i + 1)


x = [0] * 3
SAT_Table(x, 0)


[0, 0, 0]
[0, 0, 1]
[0, 1, 0]
[0, 1, 1]
[1, 0, 0]
[1, 0, 1]
[1, 1, 0]
[1, 1, 1]


`for x[i] in range(2)` is shorthand for `for value in range(2): x[i] = value`.
The list `x` is created **once** and travels through the whole recursion; each
level of the recursion owns exactly one cell of it. That is why the code is so
short. Keep this in mind: the same trick is the source of the bug in Section 5.


## 3. First attempt at *returning* the rows: a classic bug

Printing is not enough: we want to *use* the rows (for example to score each
clustering). The natural change is to return the row instead of printing it.


In [3]:
def SAT_Table_wrong(x, i):
    if i == len(x):
        return x
    else:
        for x[i] in range(2):
            return SAT_Table_wrong(x, i + 1)


print(SAT_Table_wrong([0, 0, 0], 0))


[0, 0, 0]


Only one row appears, `[0, 0, 0]`, not eight. Why?

`return` **ends the function**. The first time the loop body runs, it executes
`return SAT_Table_wrong(x, 1)`, and when that call comes back the function is
over: the assignment `x[0] = 1` is never reached. A function that must produce
*many* answers cannot return after the first one.

There are two standard repairs: collect the answers in a list, or turn the
function into a generator. Both are shown below.


## 4. Repair A: collect the rows in a list


In [4]:
def SAT_Table_list(x, i, rows):
    if i == len(x):
        rows.append(list(x))        # list(x) makes a copy -- see Section 5
    else:
        for x[i] in range(2):
            SAT_Table_list(x, i + 1, rows)
    return rows


rows = SAT_Table_list([0, 0, 0], 0, [])
print(rows)
print("number of rows:", len(rows))


[[0, 0, 0], [0, 0, 1], [0, 1, 0], [0, 1, 1], [1, 0, 0], [1, 0, 1], [1, 1, 0], [1, 1, 1]]
number of rows: 8


This works. It has two small annoyances: the extra parameter `rows` must be
carried through every call, and it is very easy to write `rows.append(x)`
instead of `rows.append(list(x))`. That mistake deserves its own section.


## 5. The aliasing trap: eight rows that are all the same

Look carefully at the next function. The only difference from the previous one
is that it appends `x` instead of `list(x)`.


In [5]:
def SAT_Table_ref(x, i, rows):
    if i == len(x):
        rows.append(x)              # no copy: a reference to x is stored
    else:
        for x[i] in range(2):
            SAT_Table_ref(x, i + 1, rows)
    return rows


rows = SAT_Table_ref([0, 0, 0], 0, [])
print("number of rows:", len(rows))
for r in rows:
    print(r)


number of rows: 8
[1, 1, 1]
[1, 1, 1]
[1, 1, 1]
[1, 1, 1]
[1, 1, 1]
[1, 1, 1]
[1, 1, 1]
[1, 1, 1]


In [6]:
print("distinct objects stored:", len({id(r) for r in rows}))


distinct objects stored: 1


Eight rows were appended, but there is only **one** list object: every entry of
`rows` is a reference to the same `x`, and when the recursion finished, `x` held
the last row it visited, `[1, 1, 1]`. All eight saved "rows" therefore print
`[1, 1, 1]`.

This is the most common bug in recursive enumeration, and it is invisible when
we print inside the function (Section 2), because there we look at `x` at the
moment when it is correct. The bug appears only when we *store* the value.

Rule: when a function writes into a shared buffer, store a **copy**
(`list(x)`, `x.copy()`, `tuple(x)`), not the buffer itself.


## 6. Repair B: a generator (`yield`)

The second repair is nicer: let the function *produce* values one at a time. A
function that contains `yield` becomes a **generator**. Calling it does not run
the body; it returns an object that produces one value each time it is asked.
`yield` is like `return`, except that the function is not destroyed: it is
suspended and continues from the same place at the next request.

`yield from` delegates to another generator: every value that the recursive call
produces is passed on.


In [7]:
def SAT_Table_gen(x, i):
    """Generate every truth-table row that agrees with x on the first i entries."""
    if i == len(x):
        yield list(x)               # a copy, so the caller may keep it
    else:
        for x[i] in range(2):
            yield from SAT_Table_gen(x, i + 1)


for row in SAT_Table_gen([0, 0, 0], 0):
    print(row)


[0, 0, 0]
[0, 0, 1]
[0, 1, 0]
[0, 1, 1]
[1, 0, 0]
[1, 0, 1]
[1, 1, 0]
[1, 1, 1]


The generator is *lazy*: values are produced on demand, so we can stop early.


In [8]:
gen = SAT_Table_gen([0, 0, 0], 0)
print(next(gen))
print(next(gen))
print("the generator is now suspended in the middle of the recursion")


[0, 0, 0]
[0, 0, 1]
the generator is now suspended in the middle of the recursion


## 7. How many rows? $2^n$

For $n$ Boolean variables the truth table has exactly $2^n$ rows, one for each
subset of the variables that are set to 1. Counting the generated rows confirms
it, and shows how quickly enumeration becomes hopeless:


In [9]:
for N in range(1, 13):
    count = sum(1 for _ in SAT_Table_gen([0] * N, 0))
    print(f"n = {N:>2}   rows = {count:>6}   2**n = {2 ** N:>6}")


n =  1   rows =      2   2**n =      2
n =  2   rows =      4   2**n =      4
n =  3   rows =      8   2**n =      8
n =  4   rows =     16   2**n =     16
n =  5   rows =     32   2**n =     32
n =  6   rows =     64   2**n =     64
n =  7   rows =    128   2**n =    128
n =  8   rows =    256   2**n =    256
n =  9   rows =    512   2**n =    512
n = 10   rows =   1024   2**n =   1024
n = 11   rows =   2048   2**n =   2048
n = 12   rows =   4096   2**n =   4096


For $n=30$ this is about $10^9$ rows and for $n=60$ about $10^{18}$. Enumeration
stops being an option long before that. For *clusterings* the count is not
$2^n$, and getting that count right is the subject of the rest of the notebook.


## 8. Counting clusterings: Stirling numbers of the second kind

A truth-table row is a *labeled* assignment: bit $i$ says which of two groups
point $i$ belongs to. A **clustering** is a *partition*: a family of non-empty
blocks with no names attached. The number of partitions of $n$ distinct objects
into exactly $k$ non-empty blocks is the **Stirling number of the second kind**

$$
S(n,k) = \left\{ \begin{matrix} n \\ k \end{matrix} \right\}.
$$

> **Mind the name.** *Stirling's formula* is the approximation
> $n! \approx \sqrt{2\pi n}\,(n/e)^n$ for the factorial. *Stirling numbers* are
> something else: counts of permutations (first kind) and of partitions (second
> kind). Only the second kind is needed here.

Why the recurrence

$$S(n,k) = k\,S(n-1,k) + S(n-1,k-1)$$

holds: look at object $n$.

* It is **alone** in its block. The other $n-1$ objects form a partition into
  $k-1$ blocks: $S(n-1,k-1)$ ways.
* It is **not alone**. Delete it: the other $n-1$ objects form a partition into
  $k$ blocks ($S(n-1,k)$ ways), and object $n$ may join any of those $k$ blocks:
  $k\,S(n-1,k)$ ways.

The two cases are disjoint and cover everything, so we add them. The base cases
are $S(n,1)=1$ (one block), $S(n,n)=1$ (one object per block) and $S(n,k)=0$ for
$k>n$ (impossible).


In [10]:
def S(n, k):
    """Stirling number of the second kind: partitions of n objects into k blocks."""
    if k == 0:
        return 1 if n == 0 else 0
    if k > n:
        return 0
    if k == 1 or k == n:
        return 1
    return k * S(n - 1, k) + S(n - 1, k - 1)


print("S(3, 2) =", S(3, 2))
print("S(4, 2) =", S(4, 2))
print("S(15, 3) =", S(15, 3))


S(3, 2) = 3
S(4, 2) = 7
S(15, 3) = 2375101


$S(3,2)=3$ is easy to check by hand: the partitions of $\{1,2,3\}$ into two
blocks are $\{1\}|\{2,3\}$, $\{2\}|\{1,3\}$ and $\{3\}|\{1,2\}$. The table below
gives the rest. Note the last column: the row sums are the **Bell numbers**
$B_n$, the number of partitions into any number of blocks.


In [11]:
W = 8
print(f"{'n':>2} | " + " ".join(f"{k:>{W}}" for k in range(1, 9)) + " | " + f"{'Bell':>10}")
print("-" * 95)
for n in range(1, 9):
    row = [S(n, k) if k <= n else None for k in range(1, 9)]
    cells = [f"{v:>{W}}" if v is not None else " " * W for v in row]
    total = sum(v for v in row if v is not None)
    print(f"{n:>2} | " + " ".join(cells) + " | " + f"{total:>10}")


 n |        1        2        3        4        5        6        7        8 |       Bell
-----------------------------------------------------------------------------------------------
 1 |        1                                                                |          1
 2 |        1        1                                                       |          2
 3 |        1        3        1                                              |          5
 4 |        1        7        6        1                                     |         15
 5 |        1       15       25       10        1                            |         52
 6 |        1       31       90       65       15        1                   |        203
 7 |        1       63      301      350      140       21        1          |        877
 8 |        1      127      966     1701     1050      266       28        1 |       4140


The Bell numbers $1, 2, 5, 15, 52, 203, 877, 4140, \ldots$ grow quickly: for
$n=10$ points there are already $115\,975$ clusterings, which is why exhaustive
search is possible for toy data sets only.


## 9. A closed form

The recurrence is not the only formula. Counting *surjective*
(label-preserving) assignments gives

$$
S(n,k) = \frac{1}{k!}\sum_{j=0}^{k}(-1)^j\binom{k}{j}(k-j)^n .
$$

The sum counts the assignments that use $k$ labels and hit all of them
(inclusion and exclusion over the labels that are missed); dividing by $k!$
forgets the names of the labels. Checking that the two formulas agree is a good
test of both:


In [12]:
import math


def S_closed(n, k):
    total = sum((-1) ** j * math.comb(k, j) * (k - j) ** n for j in range(k + 1))
    return total // math.factorial(k)


agree = all(S(n, k) == S_closed(n, k) for n in range(1, 12) for k in range(1, n + 1))
print("recurrence and closed form agree:", agree)


recurrence and closed form agree: True


## 10. Two identities used later

**Two clusters.** For $k=2$ there is a closed form, $S(n,2)=2^{n-1}-1$: in a
partition into two blocks, the block that contains point 1 is determined by which
of the points $2,\ldots,n$ join it ($2^{n-1}$ choices), and a subset and its
complement give the same partition, hence the division by two (the $-1$ removes
the empty subset).


In [13]:
ns = list(range(2, 13))
print("S(n, 2)      :", [S(n, 2) for n in ns])
print("2**(n-1) - 1 :", [2 ** (n - 1) - 1 for n in ns])


S(n, 2)      : [1, 3, 7, 15, 31, 63, 127, 255, 511, 1023, 2047]
2**(n-1) - 1 : [1, 3, 7, 15, 31, 63, 127, 255, 511, 1023, 2047]


**Labelings versus partitions.** Choosing one of $k$ labels for each of $n$
points gives $k^n$ labelings. Such a labeling uses some $j\le k$ labels
($\binom{k}{j}$ choices of which), and the points are split into $j$ non-empty
blocks in $S(n,j)$ ways; the $j$ labels can be given to those blocks in $j!$
ways. Hence

$$k^n = \sum_{j=1}^{k}\binom{k}{j}\, j!\, S(n,j).$$


In [14]:
n, k = 7, 4
print("k**n                =", k ** n)
print("sum over partitions =", sum(math.comb(k, j) * math.factorial(j) * S(n, j)
                                  for j in range(1, k + 1)))


k**n                = 16384
sum over partitions = 16384


## 11. How large is a clustering problem?

Put numbers on the story. Sorting 150 points into 3 clusters:


In [15]:
print("S(150, 3) =", S(150, 3))
print("number of digits:", len(str(S(150, 3))))


S(150, 3) = 61664747505854495487450129694992261011432242869144177577783166025584730
number of digits: 71


In [16]:
per_second = 10 ** 9                      # one billion clusterings scored per second
years = S(150, 3) / per_second / (3600 * 24 * 365)
print(f"time needed: {years:.2e} years")


time needed: 1.96e+54 years


The age of the universe is about $1.4\times10^{10}$ years, so this is hopeless by
a factor of about $10^{44}$. And $k=3$ is the *easy* case: the total number of
clusterings of 150 points, into any number of clusters, is the Bell number
$B_{150}$, which has about 150 digits.

This is the quantitative reason why every clustering chapter, after stating the
problem, turns immediately to heuristics (k-means, hierarchical clustering,
LVQ). The exhaustive count is still worth having: it *defines* the problem, and
for $n=10$ or $15$ it gives the true optimum against which a heuristic can be
tested, which is the job of the [BF-Clustering](BF-Clustering.ipynb) notebook.


## 12. Why the plain recursion is slow: overlapping subproblems

`S` is short and correct, but redundant. The calls `S(n-1,k)` and `S(n-1,k-1)`
both eventually need `S(n-2,k-1)`, and each of them computes it again. Count how
many times the function is entered:


In [17]:
calls = 0


def S_count(n, k):
    global calls
    calls += 1
    if k > n:
        return 0
    if k == 1 or k == n:
        return 1
    return k * S_count(n - 1, k) + S_count(n - 1, k - 1)


for n, k in [(10, 3), (15, 4), (16, 6), (18, 8), (20, 10)]:
    calls = 0
    value = S_count(n, k)
    print(f"S({n}, {k}) = {value:>20}   function calls = {calls:>7}")


S(10, 3) =                 9330   function calls =      71
S(15, 4) =             42355950   function calls =     727
S(16, 6) =           2734926558   function calls =    6005
S(18, 8) =         189036065010   function calls =   38895
S(20, 10) =        5917584964655   function calls =  184755


For $S(20,10)$ there are only about $20\times 10 = 200$ *different* subproblems
$(n,k)$, yet the recursion is entered 184 755 times. The cost is exponential in
$k$, not in $n$.

**Memoization** stores the answer of each subproblem the first time it is
computed. In Python that is one line:


In [18]:
from functools import lru_cache


@lru_cache(maxsize=None)
def S_memo(n, k):
    if k == 0:
        return 1 if n == 0 else 0
    if k > n:
        return 0
    if k == 1 or k == n:
        return 1
    return k * S_memo(n - 1, k) + S_memo(n - 1, k - 1)


S_memo.cache_clear()
print("S_memo(20, 10) =", S_memo(20, 10))
print("distinct subproblems computed:", S_memo.cache_info().currsize)
print("S_memo(150, 3) =", S_memo(150, 3))


S_memo(20, 10) = 5917584964655
distinct subproblems computed: 109
S_memo(150, 3) = 61664747505854495487450129694992261011432242869144177577783166025584730


In [19]:
import time

t0 = time.perf_counter()
S(21, 10)
t1 = time.perf_counter()
S_memo.cache_clear()
t2 = time.perf_counter()
S_memo(21, 10)
t3 = time.perf_counter()
print(f"without memoization: {t1 - t0:.3f} s")
print(f"with memoization:    {t3 - t2:.6f} s")


without memoization: 0.025 s
with memoization:    0.000077 s


The same idea appears all over this course: dynamic programming, the Viterbi
algorithm and the "cache" of a recursive search are all memoized recursions. The
price is memory: one entry per subproblem.

## 13. What this has to do with clustering

* A clustering is a **partition**, and the number of partitions of $n$ points
  into $k$ non-empty clusters is $S(n,k)$. The number of *labelings* is $k^n$;
  the factor $k^n/S(n,k)\approx k!$ is the relabeling symmetry, and labelings
  with an empty cluster are not clusterings at all.
* Two clusters are special: $S(n,2)=2^{n-1}-1$, which grows like $2^n$.
* For $k$ clusters and $n$ points, enumeration costs at least $S(n,k)$
  operations. For a $200\times 300$ image, $n=60\,000$; no computer will ever
  enumerate that.

Writing the enumeration once, by hand, is nevertheless worth the effort: it is
the definition of the clustering problem, and for small $n$ it produces the
optimum against which k-means is judged.

Next: the [N-Queen](N-Queen.ipynb) notebook adds **constraints** to the same
recursion, so that most of the tree is never visited, and the
[BF-Clustering](BF-Clustering.ipynb) notebook uses the recursion to solve a
clustering problem exactly.

## Exercises

1. Change `SAT_Table` so that every variable takes the values $0,1,2$. How many
   rows does it produce for $n=4$? How many in general?
2. `SAT_Table_ref` stores eight references to one list. Repair it in two
   different ways (a copy, and a generator) and check that the eight rows are
   different objects.
3. Prove that $S(n,2)=2^{n-1}-1$ using the block that contains point 1.
4. The Bell numbers satisfy $B_{n+1}=\sum_{j=0}^{n}\binom{n}{j}B_j$. Prove it
   (classify partitions by the block that contains point $n+1$) and check it
   numerically for $n\le 10$.
5. How many clusterings of 20 points into 2 clusters are there? How long would
   scoring them take at $10^6$ per second? And for 40 points at $10^9$ per
   second?
6. Use the closed form to compute $S(150,3)$ and compare it with the recursive
   value. Which of the two would you use for $n=10^4$?
